# 🏥 Medical Report Extractor – Google Colab GPU Runner

This notebook runs the staged medical extraction pipeline using **Google Colab's free GPU (T4 / A100)**.

### Step 0: Ensure GPU is Enabled
Go to **Runtime** > **Change runtime type** > Select **T4 GPU** (or A100 if Colab Pro) > Click **Save**.

In [ ]:
# Check GPU Availability
!nvidia-smi

### Step 1: Clone Your Repository

In [ ]:
!git clone https://github.com/soum-tech/Medical.git
%cd Medical

### Step 2: Install Project & GPU Dependencies

In [ ]:
!pip install --upgrade pip
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install "transformers>=5.3.0" "accelerate>=1.0.0" pymupdf Pillow pydantic python-dotenv huggingface_hub
!pip install -e .

### Step 3: Download Qwen3.5-0.8B Weights Directly to Colab
Colab has high-speed network connections, so downloading takes ~15–30 seconds.

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading Qwen3.5-0.8B model weights...")
snapshot_download(
    repo_id="Qwen/Qwen3.5-0.8B",
    local_dir="models/Qwen3.5-0.8B",
)
print("Download complete!")

### Step 4: Configure Local GPU Execution in `.env`

In [ ]:
env_content = """BACKEND=local
MODEL_PATH=models/Qwen3.5-0.8B
MODEL_DEVICE_MAP=auto
MODEL_LOCAL_FILES_ONLY=true
MODEL_MAX_NEW_TOKENS=4096
RENDER_DPI=144
MAX_PAGES_PER_CALL=1
MAX_TARGETED_RETRIES=1
"""

with open(".env", "w", encoding="utf-8") as f:
    f.write(env_content)

print(".env configured for Local GPU!")

### Step 5: Upload Your Medical Report (or use a sample in `input/`)
Run this cell if you want to upload a file directly from your computer into Colab.

In [ ]:
from google.colab import files
import os

os.makedirs("input", exist_ok=True)
print("Upload your medical PDF or image report:")
uploaded = files.upload()

input_file_path = None
for filename in uploaded.keys():
    destination = os.path.join("input", filename)
    os.replace(filename, destination)
    input_file_path = destination
    print(f"Saved to {destination}")

### Step 6: Run Extraction with the Local Model on GPU!

In [ ]:
import glob

# Auto-detect the uploaded input file, or fallback to the first file in input/
if not input_file_path or not os.path.exists(input_file_path):
    files_in_input = glob.glob("input/*")
    if files_in_input:
        input_file_path = files_in_input[0]
    else:
        raise FileNotFoundError("Please upload a report into input/ before running.")

print(f"Processing report: {input_file_path} on GPU...")
!python -m medical_report_extractor.cli --input "$input_file_path" --output-dir "output"

### Step 7: View the Dashboard in Colab & Download Results

In [ ]:
import IPython
from google.colab import files

# 1. Display the interactive review dashboard right inside the Colab notebook
IPython.display.display(IPython.display.HTML(filename="output/dashboard.html"))

In [ ]:
# 2. Zip and download output files to your local computer
!zip -r extraction_output.zip output/
files.download("extraction_output.zip")